# Vistas y window functions

Continuación de la clase de SQL avanzado sobre `ecommerce`.

En la sesión anterior medimos el negocio (`COUNT`, `SUM`, `AVG`) y lo partimos por país y ciudad (`JOIN` + `GROUP BY`). Eso **resume**. Hoy resolvemos dos problemas que `GROUP BY` no cubre bien:

1. **No reescribir el mismo JOIN** cada vez → **vistas**
2. **Comparar filas entre sí sin perder el detalle** → **window functions**

| Minutos | Bloque |
|---|---|
| 0–3 | Conexión |
| 3–18 | Vistas |
| 18–38 | Window functions |
| 38–40 | Cierre y práctica |

## 0. Conexión

La base ya está cargada. Solo conectamos y reutilizamos `consultar()`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "db" / "connection.py").exists():
    ROOT = Path(r"C:\Programacion\DataScienceIA\Tutor_Dev_Senior_Code\Tutorias_Cohorte_6\Tutorias_Refuerzo_IA6\Tutoria_persistencia_consultas_avanzadas")

sys.path.insert(0, str(ROOT / "db"))
from connection import get_connection  # pyright: ignore[reportMissingImports]

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

conn = get_connection()
conn.autocommit = True

def consultar(sql, params=None):
    """Ejecuta SQL y devuelve un DataFrame."""
    with conn.cursor() as cur:
        cur.execute(sql, params)
        if cur.description is None:
            return None
        columnas = [col[0] for col in cur.description]
        filas = cur.fetchall()
    return pd.DataFrame(filas, columns=columnas)

consultar("SELECT current_database() AS db, current_user AS usuario")

## 1. Vistas (15 min)

Una **vista** es una consulta guardada con nombre. No guarda filas: guarda el SQL. Cada vez que la lees, PostgreSQL vuelve a ejecutarla sobre los datos actuales.

| | Tabla | Vista |
|---|---|---|
| Qué guarda | datos | la consulta |
| Se actualiza | al hacer `INSERT`/`UPDATE` | al consultarla (ve datos frescos) |
| Para qué | persistir hechos | no repetir JOINs ni reglas de negocio |

En la clase anterior el JOIN `ventas + clientes` apareció más de una vez. Eso es señal de vista.

### 1.1 Crear la vista del ticket enriquecido

Pregunta de negocio: *¿cómo dejo listo el detalle venta–cliente–producto para no armar el JOIN en cada pregunta?*

In [ ]:
consultar("""
CREATE OR REPLACE VIEW v_ventas_enriquecidas AS
SELECT
    v.venta_id,
    v.fecha_venta,
    v.cantidad,
    v.total,
    v.canal,
    c.cliente_id,
    c.nombre   AS cliente,
    c.pais,
    c.ciudad,
    c.segmento,
    p.producto_id,
    p.nombre   AS producto,
    p.categoria
FROM ventas v
JOIN clientes  c ON c.cliente_id  = v.cliente_id
JOIN productos p ON p.producto_id = v.producto_id
""")

print("Vista creada: v_ventas_enriquecidas")

A partir de aquí se consulta **como si fuera una tabla**. La diferencia: no hay datos copiados, solo el plan de la consulta.

In [ ]:
consultar("""
-- Misma pregunta geográfica de la clase anterior,
-- pero el JOIN ya vive dentro de la vista.
SELECT
    pais,
    ciudad,
    COUNT(DISTINCT cliente_id) AS n_clientes,
    COUNT(*)                   AS n_ventas,
    ROUND(SUM(total), 2)       AS ingresos
FROM v_ventas_enriquecidas
GROUP BY pais, ciudad
ORDER BY ingresos DESC
""")

### 1.2 Vista de KPIs mensuales

Una vista también puede **agregar**. Útil cuando el mismo resumen se usa en varias preguntas (evolución, comparativos, ranking de meses).

In [ ]:
consultar("""
CREATE OR REPLACE VIEW v_kpis_mensuales AS
SELECT
    DATE_TRUNC('month', fecha_venta)::date AS mes,
    COUNT(*)                     AS n_ventas,
    COUNT(DISTINCT cliente_id)   AS clientes,
    ROUND(SUM(total), 2)         AS ingresos,
    ROUND(AVG(total), 2)         AS ticket_medio
FROM ventas
GROUP BY DATE_TRUNC('month', fecha_venta)
""")

consultar("""
SELECT *
FROM v_kpis_mensuales
ORDER BY mes
LIMIT 8
""")

### 1.3 Cómo se ve en el catálogo

La vista queda en la base. Cualquier sesión posterior puede usarla. `CREATE OR REPLACE` la actualiza si cambiamos la definición.

In [ ]:
consultar("""
SELECT table_name AS vista
FROM information_schema.views
WHERE table_schema = 'public'
ORDER BY 1
""")

**Reglas rápidas para la pizarra**

- `CREATE VIEW` / `CREATE OR REPLACE VIEW` / `DROP VIEW`
- Se lee con `SELECT ... FROM nombre_vista`
- No sustituye una tabla: si el JOIN es pesado y se consulta mil veces al día, más adelante aparece la **vista materializada** (guarda el resultado). Hoy no la necesitamos.
- Si cambian `ventas`, `clientes` o `productos`, la vista ya refleja el cambio en la siguiente lectura.

## 2. Window functions (20 min)

`GROUP BY` **colapsa** filas: 200.000 ventas → 12 ciudades. Una **window function** calcula sobre un conjunto de filas **y deja cada fila viva**.

Anatomía:

```sql
FUNCION(...) OVER (
    PARTITION BY grupo    -- el "GROUP BY" de la ventana (opcional)
    ORDER BY criterio     -- orden dentro del grupo (a veces obligatorio)
)
```

| Necesitas | `GROUP BY` | Window |
|---|---|---|
| Un total por país | sí | no hace falta |
| Cada venta + el promedio de su país | no | sí |
| Top 3 productos por categoría | se complica | `RANK()` |
| Ingreso acumulado mes a mes | no | `SUM() OVER (ORDER BY mes)` |
| ¿Crecimos vs el mes anterior? | no | `LAG()` |

### 2.1 El contraste: detalle + promedio del grupo

Pregunta: *de cada venta, ¿está por encima o por debajo del ticket medio de su país?*

Con solo `GROUP BY` perderíamos la venta. Con `AVG(...) OVER (PARTITION BY pais)` no.

In [ ]:
consultar("""
-- PARTITION BY pais = "calcula el promedio dentro de cada país".
-- La fila de la venta no desaparece.
SELECT
    venta_id,
    pais,
    cliente,
    total,
    ROUND(AVG(total) OVER (PARTITION BY pais), 2) AS ticket_medio_pais,
    ROUND(total - AVG(total) OVER (PARTITION BY pais), 2) AS vs_pais
FROM v_ventas_enriquecidas
ORDER BY venta_id
LIMIT 12
""")

### 2.2 Ranking: top 3 productos por categoría

Pregunta: *¿cuáles son los 3 productos que más ingresan en cada categoría?*

Tres funciones parecidas; la diferencia es cómo tratan los empates:

| Función | Empate | Siguiente número |
|---|---|---|
| `ROW_NUMBER()` | no empata: fuerza 1, 2, 3... | sigue |
| `RANK()` | 1, 1, 3 | salta el 2 |
| `DENSE_RANK()` | 1, 1, 2 | no salta |

In [ ]:
consultar("""
-- 1) Agregamos ingresos por producto (aquí sí usamos GROUP BY).
-- 2) RANK() parte por categoría y ordena por ingresos.
-- 3) Nos quedamos con los 3 primeros de cada categoría.
WITH por_producto AS (
    SELECT
        categoria,
        producto,
        ROUND(SUM(total), 2) AS ingresos,
        RANK() OVER (
            PARTITION BY categoria
            ORDER BY SUM(total) DESC
        ) AS ranking
    FROM v_ventas_enriquecidas
    GROUP BY categoria, producto
)
SELECT *
FROM por_producto
WHERE ranking <= 3
ORDER BY categoria, ranking
""")

### 2.3 Acumulado: ingreso mes a mes

Pregunta: *¿cómo crece el ingreso acumulado a lo largo del tiempo?*

`SUM(...) OVER (ORDER BY mes)` convierte cada mes en "yo + todos los anteriores". Sin `PARTITION BY`, la ventana es **toda la serie**.

In [ ]:
consultar("""
-- SUM(SUM(total)) se lee así:
--   el SUM interno es el GROUP BY del mes,
--   el SUM externo recorre esos meses en orden y va acumulando.
SELECT
    mes,
    ingresos,
    ROUND(
        SUM(ingresos) OVER (ORDER BY mes),
        2
    ) AS ingresos_acumulados
FROM v_kpis_mensuales
ORDER BY mes
""")

### 2.4 LAG: ¿crecimos respecto al mes anterior?

Pregunta: *¿qué meses subieron o bajaron contra el mes previo?*

`LAG(columna) OVER (ORDER BY mes)` trae el valor de la **fila anterior**. `LEAD` trae la siguiente. El primer mes no tiene anterior → `NULL`.

In [ ]:
consultar("""
SELECT
    mes,
    ingresos,
    LAG(ingresos) OVER (ORDER BY mes) AS mes_anterior,
    ROUND(
        ingresos - LAG(ingresos) OVER (ORDER BY mes),
        2
    ) AS delta,
    ROUND(
        100.0 * (ingresos - LAG(ingresos) OVER (ORDER BY mes))
        / NULLIF(LAG(ingresos) OVER (ORDER BY mes), 0),
        2
    ) AS var_pct
FROM v_kpis_mensuales
ORDER BY mes
""")

### 2.5 NTILE: partir clientes en cuartiles

Pregunta: *si ordenamos a los clientes por ingreso, ¿cuánto concentra el 25% de arriba?*

`NTILE(4)` reparte las filas en 4 grupos del mismo tamaño (aprox.): 1 = más altos, 4 = más bajos si ordenamos `DESC`.

In [ ]:
consultar("""
WITH cliente_ingresos AS (
    SELECT
        cliente_id,
        SUM(total) AS ingresos
    FROM ventas
    GROUP BY cliente_id
),
cuartiles AS (
    SELECT
        cliente_id,
        ingresos,
        NTILE(4) OVER (ORDER BY ingresos DESC) AS cuartil
    FROM cliente_ingresos
)
SELECT
    cuartil,
    COUNT(*) AS n_clientes,
    ROUND(MIN(ingresos), 2) AS desde,
    ROUND(MAX(ingresos), 2) AS hasta,
    ROUND(SUM(ingresos), 2) AS ingresos,
    ROUND(
        100.0 * SUM(ingresos) / SUM(SUM(ingresos)) OVER (),
        2
    ) AS pct_ingreso
FROM cuartiles
GROUP BY cuartil
ORDER BY cuartil
""")

## 3. Juntar las dos ideas (cierre)

Una vista puede **guardar una window function**. Así el ranking queda listo para el resto del equipo, igual que el JOIN.

In [ ]:
consultar("""
CREATE OR REPLACE VIEW v_top_productos_categoria AS
SELECT
    categoria,
    producto,
    ROUND(SUM(total), 2) AS ingresos,
    RANK() OVER (
        PARTITION BY categoria
        ORDER BY SUM(total) DESC
    ) AS ranking
FROM v_ventas_enriquecidas
GROUP BY categoria, producto
""")

consultar("""
SELECT *
FROM v_top_productos_categoria
WHERE ranking = 1
ORDER BY ingresos DESC
""")

## 4. Práctica (el resto de la hora)

Trabajar sobre `v_ventas_enriquecidas` y `v_kpis_mensuales`.

1. Top 5 clientes por ingreso en **Colombia**, con `RANK()` o `ROW_NUMBER()`.
2. Ingreso mensual **por canal** y variación vs el mes anterior (`PARTITION BY canal` + `LAG`).
3. Crear `v_kpis_pais` con clientes, ventas e ingresos por país, y consultarla.

In [ ]:
consultar(
    """

    """
)

In [ ]:
consultar(
    """

    """
)

In [ ]:
consultar(
    """

    """
)